%md
# Day 2: Token Management & Conversation History

---

## 💰 The Token Accumulation Problem

From Day 1, we learned that every message in a chat session uses **tokens**. But there's a hidden cost: **LLMs reprocess the entire conversation history with every new message**.

### How Token Usage Grows

#### Example Conversation:

| Turn | Message | Tokens Used | Total Tokens Processed |
| --- | --- | --- | --- |
| 1 | **User:** "Hi how are you" | 5 | **5** |
| 1 | **Assistant:** "Hi I am doing good how are you doing today?" | 10 | **15** (5 + 10) |
| 2 | **User:** "My name is Vinith" | 4 | **19** (15 + 4) |
| 2 | **Assistant:** Response | ~10 | **29** (19 + 10) |

### The Key Problem
**Every new message requires processing ALL previous tokens again**

* Turn 1: Process 5 tokens → Generate response (10 tokens) = **15 total tokens**
* Turn 2: Process **15 old tokens** + 4 new tokens = **19 tokens processed**
* Turn 3: Process **29 old tokens** + new tokens = **keeps growing**

### Why This Matters
* **Cost:** Most LLM APIs charge per token processed
* **Speed:** More tokens = slower response times
* **Limits:** APIs have maximum token limits (e.g., 4K, 8K, 128K)
* **Example:** 300 messages in history = **very expensive operation**

---

## 🔧 Attempted Solutions & Their Problems

When conversation history becomes too large, we need strategies to reduce tokens. But each approach has trade-offs:

### Strategy 1: Keep Only Last N Messages

#### Approach
* Keep only the **last 50 messages** in history
* Discard older messages

#### Problem: Loss of Early Context

**Scenario:**
```
Message 1: "My name is Vinith"
Message 2-49: Other conversation
Message 50: "What is my name?"
```

* ❌ **Fails:** Message 1 was discarded
* The LLM cannot answer because the context is no longer in history

---

### Strategy 2: Keep First N + Last N Messages

#### Approach
* Keep **first 10 messages** (early context)
* Keep **last 10 messages** (recent context)
* Discard everything in between

#### Problem: Loss of Middle Context

**Scenario:**
```
Message 1-10: Introduction and setup
Message 11-280: You discuss a project (contains important details)
Message 281-290: Current conversation
Message 291: "What did we decide about the project?"
```

* ❌ **Fails:** Project details (messages 11-280) were discarded
* The critical context is in the middle, which is missing

---

### Strategy 3: Summarization + Recent Messages

#### Approach
* **Summarize** messages 1 to (N-50)
* Keep **last 50 messages** as-is
* Send: `[Summary] + [Last 50 Messages] + [Current Message]` to LLM

#### Benefits
* Reduces tokens dramatically
* **Example:** 300,000 tokens → 3,000 tokens (100x reduction)
* Preserves recent conversation in full detail

#### Problem: Summarization Loses Details

**Scenario:**
```
Message 50: "My API key is abc123xyz"
Message 51-200: Other topics
Summary: "User discussed authentication and other topics"
Message 201: "What was my API key again?"
```

* ❌ **May Fail:** The summary didn't include the specific API key
* Summaries capture **general topics**, not **specific details**
* Your next question might need a detail that was summarized away

---

## 📊 Strategy Comparison

| Strategy | Token Savings | Pros | Cons |
| --- | --- | --- | --- |
| **Keep all messages** | None | Perfect context retention | Expensive, slow, hits limits |
| **Last N messages** | High | Simple, keeps recent context | Loses early context |
| **First N + Last N** | High | Keeps intro and recent | Loses middle context |
| **Summary + Last N** | Very High | Balances cost and context | May lose critical details |

---

## 🧩 The Core Challenge

**There is no perfect solution** for managing long conversation history:

1. **Keep everything** → Expensive and slow
2. **Discard old messages** → Lose important context
3. **Summarize** → Lose specific details

### The Trade-off Triangle

```
         Cost/Speed
            ⬆️
            |
            |
  Detail ⬅️───➡️ Context
```

You can only optimize 2 out of 3:
* **Low cost + Full context** = Lose details (summarization)
* **Low cost + Full details** = Lose context (discard old messages)
* **Full context + Full details** = High cost (keep everything)

---

## 💡 Key Takeaways

1. **Token usage grows exponentially** with conversation length
2. **Every message reprocesses the entire history** — this is expensive
3. **Keeping only recent messages** loses early context
4. **Keeping first + last messages** loses middle context  
5. **Summarization reduces tokens** but may lose critical details
6. **There is no perfect strategy** — all approaches involve trade-offs
7. The right strategy depends on your specific use case and budget

---

## 🔮 What's Next?

In future lessons, we'll explore:
* **Advanced context management** techniques
* **Retrieval-Augmented Generation (RAG)** — pulling relevant context on-demand
* **Vector databases** for semantic search of conversation history
* **Hybrid approaches** combining multiple strategies 


when you have some instutions to the model, if you give them in the chat , lets say once the model is deployed end users may delete the chat then you ill end up clearing all the chats which you will lose the intsrtution aslso ,to preserve the insturion you need to give it in such a way like configs ( sustemindtion ) in some llm modesl , in some its part of the message ( role : system ) etc , and role(user) - will be you talkign ato an agent , role: assistean is respoce from user 


so when you are calling the clear or delte chat fucntion you need to only remove the chats from 2nd msgs form the chat not from 1st , by this you will always delte from user or assister role not the system instions 


In [0]:
# Install package
%pip install groq

import sys
from groq import Groq

# Add user folder to Python path to import config
sys.path.insert(0, '/Users/vinith.singareddy@gmail.com')

# Import API key from config file (stored outside git repo)
from groq_config import GROQ_API_KEY

# Initialize client
client = Groq(api_key=GROQ_API_KEY)

# Call model
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are an elite Azure Data Engineering specialist with 30+ years of enterprise experience in data platforms, analytics architecture, cloud migrations, distributed systems, governance, and production-scale implementations.You operate as a senior architect, consultant, troubleshooter, and mentor focused only on Microsoft Azure data technologies and directly related ecosystems.Your expertise includes, but is not limited to:        Azure Data Factory (ADF)        Azure Synapse Analytics        Azure Databricks        Azure Data Lake Storage (ADLS)        Azure SQL Database        Azure SQL Managed Instance        SQL Server        Azure Fabric / Microsoft Fabric        Azure Stream Analytics        Azure Event Hubs        Azure Functions for data workloads        Logic Apps for integrations        Power BI (data modeling / pipelines / semantic layer)        Azure DevOps CI/CD for data platforms        ARM / Bicep / Terraform for Azure data infrastructure        Python for data engineering        PySpark / Spark optimization        T-SQL advanced development        Data modeling (OLTP / OLAP / Star Schema / Medallion)        ETL / ELT architecture        Incremental loads / CDC        Data governance        Security / RBAC / Managed Identity / Key Vault        Cost optimization        Performance tuning        Monitoring / Observability        Production support / root cause analysis        Behavior Rules:        Answer only Azure data engineering or directly related technical questions.        If a question is outside Azure data engineering scope, politely state: “I specialize only in Azure data engineering and related data platform topics.”        Give expert-level, practical, production-ready answers.        Prefer real-world enterprise best practices over textbook theory.        When multiple solutions exist, recommend the best option with reasoning.        Include performance, cost, security, and maintainability considerations.        Provide step-by-step guidance when troubleshooting.        Provide code when useful (SQL, Python, PySpark, ARM, JSON, ADF expressions, etc.).        Explain beginner to advanced depending on user question.        Never give vague answers—be precise and actionable.        Assume mission-critical production environments.        If information is uncertain, clearly state assumptions.        Response Style:        Clear, direct, technical.        Concise first, detailed when needed.        Use bullet points for architecture or troubleshooting.        Use examples from Azure enterprise environments.        Focus on solving the problem fast and correctly.        Primary Identity:        You are the final escalation-level Azure Data Engineering expert with deep hands-on knowledge across Microsoft Azure data services."},
        {"role": "user", "content": "how to use logicapps and call an from adf to send email help me in detailed "}
    ],
    temperature=0.7,  # If set, partial message deltas will be sent.
    stream=True,
)

# Print the incremental deltas returned by the LLM.
for chunk in response:
    print(chunk.choices[0].delta.content, end="")